# 🛠️ Step 1: Professional Data Preprocessing Pipeline
**Objective:** Clean, transform, and prepare raw data for Machine Learning models without causing Data Leakage. We will use `scikit-learn` Pipelines and `ColumnTransformer` for an industry-standard workflow.

In [8]:
# Import Essential Libraries
import pandas as pd
import numpy as np

# Scikit-Learn modules for preprocessing and pipelines
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder




### 📂 Step 2: Load the Dataset
For this template, we will create a small synthetic dataset. In a real project, you would replace this with `pd.read_csv('your_file.csv')`.

In [9]:
# Creating a dummy dataset to test the pipeline
data = {
    'Age': [25, np.nan, 30, 35, 40, 25],          # Numerical with missing value
    'Income': [50000, 60000, 55000, np.nan, 70000, 50000], # Numerical with missing value
    'City': ['Cairo', 'Alex', 'Cairo', 'Giza', np.nan, 'Cairo'], # Categorical with missing value
    'Purchased': ['Yes', 'No', 'Yes', 'No', 'Yes', 'Yes']        # Target variable
}

df = pd.DataFrame(data)

# Display the raw data
print("Raw Dataset:")
display(df)

Raw Dataset:


,Age,Income,City,Purchased
0,25.0,50000.0,Cairo,Yes
1,NaN,60000.0,Alex,No
2,30.0,55000.0,Cairo,Yes
3,35.0,NaN,Giza,No
4,40.0,70000.0,NaN,Yes
5,25.0,50000.0,Cairo,Yes


### 🧹 Step 3: Basic Data Cleaning
Before applying complex transformations, we handle basic structural issues like duplicate rows.

In [10]:
# Check for duplicates and drop them
duplicates_count = df.duplicated().sum()
print(f"Found {duplicates_count} duplicate rows. Dropping them...")

df.drop_duplicates(inplace=True)

# Reset index after dropping rows
df.reset_index(drop=True, inplace=True)

Found 1 duplicate rows. Dropping them...



### 🎯 Step 4: Separate Features (X) and Target (y)
We must separate the data we want to predict (`y`) from the data we will use to make the prediction (`X`).

In [11]:
# Separate features and target
X = df.drop('Purchased', axis=1) # Features
y = df['Purchased']              # Target

# Automatically identify numerical and categorical columns
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

print(f"Numerical Features: {list(numerical_cols)}")
print(f"Categorical Features: {list(categorical_cols)}")

Numerical Features: ['Age', 'Income']
Categorical Features: ['City']


### 🏗️ Step 5: Build the Preprocessing Pipelines
We create separate pipelines for numerical and categorical data to handle their specific needs (like replacing missing values and scaling/encoding).

In [12]:
# 1. Pipeline for Numerical Features
# - Impute missing values with the 'mean'
# - Scale data using StandardScaler
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 2. Pipeline for Categorical Features
# - Impute missing values with the most frequent category (mode)
# - Apply One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# 3. Combine both pipelines using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])




### ✂️ Step 6: Train-Test Split
Always split the data **before** applying transformations to prevent Data Leakage (where the model learns information from the test set).

In [13]:
# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: X={X_train.shape}, y={y_train.shape}")
print(f"Testing data shape: X={X_test.shape}, y={y_test.shape}")

Training data shape: X=(4, 3), y=(4,)
Testing data shape: X=(1, 3), y=(1,)



### 🚀 Step 7: Execute the Pipeline
We use `fit_transform` on the **training data** (to learn the patterns and transform it).
We strictly use only `transform` on the **testing data** (to apply the learned patterns without peeking at the answers).

In [14]:
# Apply the pipeline to the training data
X_train_processed = preprocessor.fit_transform(X_train)

# Apply the pipeline to the testing data
X_test_processed = preprocessor.transform(X_test)

# Display the final processed array shape
print(f"Processed X_train shape: {X_train_processed.shape}")
print(f"Processed X_test shape: {X_test_processed.shape}")

# (Optional) Retrieve feature names after one-hot encoding
try:
    cat_features = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_cols)
    all_features = list(numerical_cols) + list(cat_features)
    print("\nFinal Feature Names after Encoding:")
    print(all_features)
except Exception as e:
    pass

Processed X_train shape: (4, 4)
Processed X_test shape: (1, 4)

Final Feature Names after Encoding:
['Age', 'Income', 'City_Cairo', 'City_Giza']
